In [2]:
import pandas as pd
import os
from pathlib import Path
from chronos import ChronosPipeline
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [ ]:
pipeline = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="cuda")

def predict(context_df, test_df, pipeline):
  pred_df = pipeline.predict_df(
      context_df,
      prediction_length=len(test_df),
      quantile_levels=[0.1, 0.5, 0.9],
      id_column="item_id",
      timestamp_column="timestamp",
      target="target",
  )

  return pred_df

In [ ]:

def plot(context_df, pred_df, test_df, title=None, save_path=None):
  prediction_length = len(pred_df)
  ts_context = context_df.set_index("timestamp")["target"].tail(24)
  ts_pred = pred_df.set_index("timestamp")
  ts_ground_truth = test_df.set_index("timestamp")["target"].head(prediction_length)

  fig, ax = plt.subplots(figsize=(12, 3))
  ts_context.plot(ax=ax, label="historical data", color="xkcd:azure")
  ts_ground_truth.plot(ax=ax, label="future data (ground truth)", color="xkcd:grass green")
  ts_pred["predictions"].plot(ax=ax, label="forecast", color="xkcd:violet")
  ax.fill_between(
      ts_pred.index,
      ts_pred["0.1"],
      ts_pred["0.9"],
      alpha=0.7,
      label="prediction interval",
      color="xkcd:light lavender",
  )
  if title is not None:
      ax.set_title(title)
  ax.legend()
  fig.tight_layout()

  if save_path is not None:
      save_path = Path(save_path)
      save_path.parent.mkdir(parents=True, exist_ok=True)
      fig.savefig(save_path, dpi=150, bbox_inches="tight")
      plt.close(fig)
  else:
      plt.show()

In [6]:

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

def evaluate_predictions(test_df, prediction_df):
    evaluation_df = test_df.merge(
        prediction_df[["timestamp", "predictions"]],
        on="timestamp",
        how="inner",
    )

    actual = evaluation_df["target"].values
    predicted = evaluation_df["predictions"].values

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)
    bias = (actual - predicted).mean()

    # MAPE: sklearn's version skips zero actuals
    mape = mean_absolute_percentage_error(actual, predicted) * 100

    # SMAPE: not in sklearn, compute manually
    denom = (np.abs(actual) + np.abs(predicted))
    smape = (np.where(denom == 0, 0, 2 * np.abs(actual - predicted) / denom)).mean() * 100

    metrics = {
        "n_points": len(evaluation_df),
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "smape": smape,
        "bias": bias,
        "r2": r2,
    }
    return evaluation_df, metrics


HORIZONS = list(range(1, 11))  # 1hr through 10hr
MAX_HORIZON = 10

results_root = Path("results")
results_root.mkdir(exist_ok=True)

summaries_dir = results_root / "summaries"
summaries_dir.mkdir(exist_ok=True)

all_metrics_rows = []
zone_summary_rows = []

data_dir = Path("data")
for zone in sorted(data_dir.glob("*")):
    if not zone.is_dir():
        continue

    zone_results_dir = results_root / zone.name
    zone_results_dir.mkdir(parents=True, exist_ok=True)
    zone_metrics_rows = []
    zone_txt_lines = [f"Zone: {zone.name}\n{'='*40}\n"]

    for csv_file in sorted(zone.glob("*.csv")):
        df = pd.read_csv(csv_file, parse_dates=["timestamp"])
        df["item_id"] = csv_file.stem

        split_idx = int(len(df) * 0.97)
        if split_idx <= 0 or split_idx >= len(df):
            continue

        context_df = df.iloc[:split_idx].copy()
        test_df = df.iloc[split_idx:].copy()

        # Single predict call for all horizons
        prediction_df = pipeline.predict_df(
            context_df,
            prediction_length=MAX_HORIZON,
            quantile_levels=[0.1, 0.5, 0.9],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
        )

        # Save plot and evaluation CSV only for the full 10hr forecast
        plot_path = zone_results_dir / f"{csv_file.stem}_{MAX_HORIZON}hr.png"
        evaluation_path = zone_results_dir / f"{csv_file.stem}_{MAX_HORIZON}hr_evaluation.csv"
        plot(
            context_df,
            prediction_df,
            test_df.head(MAX_HORIZON),
            title=f"{zone.name} / {csv_file.stem} — {MAX_HORIZON}hr forecast",
            save_path=plot_path,
        )
        prediction_df.to_csv(evaluation_path, index=False)

        zone_txt_lines.append(f"Series: {csv_file.stem}\n")
        zone_txt_lines.append(f"  plot: {plot_path.name}  |  predictions: {evaluation_path.name}\n")

        for horizon in HORIZONS:
            pred_horizon_df = prediction_df.head(horizon)
            test_horizon_df = test_df.head(horizon)
            evaluation_df, metrics = evaluate_predictions(test_horizon_df, pred_horizon_df)

            row = {
                "series_id": csv_file.stem,
                "horizon_hr": horizon,
                "plot_file": plot_path.name if horizon == MAX_HORIZON else None,
                "evaluation_file": evaluation_path.name if horizon == MAX_HORIZON else None,
                **metrics,
            }
            zone_metrics_rows.append(row)
            all_metrics_rows.append({"zone": zone.name, **row})

            zone_txt_lines.append(
                f"  [{horizon}hr] MSE: {metrics['mse']:.6f}  |  RMSE: {metrics['rmse']:.6f}\n"
            )

        zone_txt_lines.append("\n")

    with open(zone_results_dir / "metrics.txt", "w") as f:
        f.writelines(zone_txt_lines)

    zone_metrics_df = pd.DataFrame(zone_metrics_rows).sort_values(["series_id", "horizon_hr"])
    zone_metrics_df.to_csv(zone_results_dir / "metrics.csv", index=False)

    numeric_cols = ["mse", "rmse", "mae", "mape", "smape", "bias", "r2"]
    zone_summary_rows_local = []
    for horizon in HORIZONS:
        subset = zone_metrics_df[zone_metrics_df["horizon_hr"] == horizon]
        zone_summary_rows_local.append({
            "zone": zone.name,
            "horizon_hr": horizon,
            "series_count": len(subset),
            **{f"mean_{col}": subset[col].mean() for col in numeric_cols},
        })

    zone_summary_df = pd.DataFrame(zone_summary_rows_local)
    zone_summary_df.to_csv(zone_results_dir / "summary.csv", index=False)
    zone_summary_df.to_csv(summaries_dir / f"{zone.name}_summary.csv", index=False)
    zone_summary_rows.extend(zone_summary_rows_local)

all_metrics_df = pd.DataFrame(all_metrics_rows).sort_values(["zone", "series_id", "horizon_hr"])
all_metrics_df.to_csv(results_root / "metrics_all_zones.csv", index=False)

all_zone_summary_df = pd.DataFrame(zone_summary_rows).sort_values(["zone", "horizon_hr"])
all_zone_summary_df.to_csv(results_root / "summary_all_zones.csv", index=False)

all_zone_summary_df

/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/sklearn/metrics/_regression.p

,zone,horizon_hr,series_count,mean_mse,mean_rmse,mean_mae,mean_mape,mean_smape,mean_bias,mean_r2
0,CR,1,272,0.727360,0.183336,0.183336,8.933919e+15,184.896720,0.139710,NaN
1,CR,2,272,0.847631,0.255209,0.196603,7.657833e+15,184.339907,0.156306,-1.258838
2,CR,3,272,0.665496,0.261918,0.181878,7.784717e+15,183.594537,0.139760,-0.407478
3,CR,4,272,0.726808,0.305606,0.196186,7.661392e+15,182.957316,0.155083,-0.288492
4,CR,5,272,0.934833,0.357018,0.221527,7.434085e+15,181.889853,0.181200,-0.214326
...,...,...,...,...,...,...,...,...,...,...
165,WR,6,424,0.093781,0.121493,0.067357,4.666386e+15,186.885228,0.043306,-0.099833
166,WR,7,424,0.102412,0.132432,0.069570,4.789060e+15,186.113164,0.044811,-0.096886
167,WR,8,424,0.105239,0.141943,0.071683,5.143629e+15,186.011642,0.045372,-0.171267
168,WR,9,424,0.230619,0.171907,0.081146,5.442507e+15,186.005610,0.053729,-0.099763


In [7]:
all_metrics_df.head()


,zone,series_id,horizon_hr,plot_file,evaluation_file,n_points,mse,rmse,mae,mape,smape,bias,r2
0,CR,0,1,NaN,NaN,1,0.000526,0.022943,0.022943,9.832735e+00,10.341144,0.022943,NaN
1,CR,0,2,NaN,NaN,2,0.014715,0.121304,0.096476,3.828255e+16,105.170572,-0.073533,-0.081078
2,CR,0,3,NaN,NaN,3,0.010377,0.101866,0.078064,3.171248e+16,136.780381,-0.062768,0.142337
3,CR,0,4,NaN,NaN,4,0.008150,0.090278,0.068135,2.810232e+16,152.585286,-0.056664,0.201613
4,CR,0,5,NaN,NaN,5,0.008887,0.094270,0.076265,3.228008e+16,162.068229,-0.067088,-0.020176


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

CSV_PATH   = Path("/home/swagnik/swagnik/chronos/results/summary_all_zones.csv")
OUTPUT_DIR = Path("/home/swagnik/swagnik/chronos/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "DejaVu Serif"],
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.labelsize":     12,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "legend.fontsize":    10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.5,
    "grid.alpha":         0.6,
    "lines.linewidth":    1.8,
    "lines.markersize":   6,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.facecolor":  "white",
})

RMSE_COLOR = "#1f77b4"   # steel blue
MAE_COLOR  = "#d62728"   # brick red
MARKER     = "o"

df    = pd.read_csv(CSV_PATH)
zones = sorted(df["zone"].unique())
print(f"Generating plots for {len(zones)} zones → {OUTPUT_DIR}/\n")

for zone in zones:
    zdf = df[df["zone"] == zone].sort_values("horizon_hr")
    x   = zdf["horizon_hr"].values

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    ax.plot(
        x, zdf["mean_rmse"].values,
        color=RMSE_COLOR, marker=MARKER,
        markerfacecolor="white", markeredgewidth=1.8,
        label="Mean RMSE",
    )
    ax.plot(
        x, zdf["mean_mae"].values,
        color=MAE_COLOR, marker=MARKER,
        markerfacecolor="white", markeredgewidth=1.8,
        linestyle="--",
        label="Mean MAE",
    )

    ax.set_xlabel("Forecast Horizon (hr)")
    ax.set_ylabel("Error")
    ax.set_title(f"Zone {zone} — RMSE & MAE vs. Forecast Horizon")
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.set_xlim(x[0] - 0.3, x[-1] + 0.3)
    ax.legend(frameon=True, framealpha=0.85, edgecolor="lightgrey")

    fig.tight_layout()
    out_path = OUTPUT_DIR / f"zone_{zone}_rmse_mae.png"
    fig.savefig(out_path)
    plt.close(fig)
    print(f"  ✓ {zone:6s}  →  {out_path.name}")

print(f"\nDone. {len(zones)} plots saved under  {OUTPUT_DIR}/")


Generating plots for 17 zones → /home/swagnik/swagnik/chronos/plots/

  ✓ CR      →  zone_CR_rmse_mae.png
  ✓ ECOR    →  zone_ECOR_rmse_mae.png
  ✓ ECR     →  zone_ECR_rmse_mae.png
  ✓ ER      →  zone_ER_rmse_mae.png
  ✓ KRCL    →  zone_KRCL_rmse_mae.png
  ✓ NCR     →  zone_NCR_rmse_mae.png
  ✓ NE      →  zone_NE_rmse_mae.png
  ✓ NFR     →  zone_NFR_rmse_mae.png
  ✓ NR      →  zone_NR_rmse_mae.png
  ✓ NWR     →  zone_NWR_rmse_mae.png
  ✓ SCR     →  zone_SCR_rmse_mae.png
  ✓ SECR    →  zone_SECR_rmse_mae.png
  ✓ SER     →  zone_SER_rmse_mae.png
  ✓ SR      →  zone_SR_rmse_mae.png
  ✓ SWR     →  zone_SWR_rmse_mae.png
  ✓ WCR     →  zone_WCR_rmse_mae.png
  ✓ WR      →  zone_WR_rmse_mae.png

Done. 17 plots saved under  /home/swagnik/swagnik/chronos/plots/
